# 01 — Inspect and prepare Egyptian car listings

Milestone 1 of the Egyptian car pricer. Same idea as Week 7 Day 2: turn raw listings into `prompt` / `completion` splits for SFT.

This notebook does **not** train a model and does **not** push to the Hugging Face Hub.

**What you will do**
1. Load [mo-hug-me/Egyptian_cars_price_prediction](https://huggingface.co/datasets/mo-hug-me/Egyptian_cars_price_prediction)
2. Parse structured fields out of the already-formatted prompt
3. Apply the v1 filter rules and watch how many rows each rule drops
4. Split train / val / test (200-row test, stratified by brand)
5. Save locally under `data/` (gitignored)

Prices stay in **EGP**. We do not scrape Hatla2ee or OLX.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
import os

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from pricer.items import PREFIX
from pricer.prep import (
    LITE_TRAIN_SIZE,
    SOURCE_DATASET,
    TEST_SIZE,
    VAL_SIZE,
    apply_hard_filters,
    apply_price_outliers,
    drop_exact_duplicates,
    lite_train,
    load_source,
    parse_frame,
    save_splits,
    split_frame,
)

load_dotenv(ROOT / ".env", override=True)
token = os.getenv("HF_TOKEN")
if token:
    login(token, add_to_git_credential=False)
    print("Logged in to Hugging Face")
else:
    print("No HF_TOKEN in .env — loading the public dataset anonymously")

## 1. Load the source dataset

The Hub file is already in Week 7 SFT shape: `prompt`, `completion`, `text`.

- **Train on** `prompt` + `completion`
- **Ignore** `text` for training (`text` is prompt + completion + `<|endoftext|>`)
- At inference the predictor must see only `prompt`, which ends with `Price is EGP`

In [ ]:
raw = load_source()
print(f"Rows: {len(raw):,}   columns: {list(raw.columns)}")
print("\n--- prompt ---")
print(raw.iloc[0]["prompt"])
print("\n--- completion ---")
print(raw.iloc[0]["completion"])
print("\nPREFIX check:", raw.iloc[0]["prompt"].rstrip().endswith(PREFIX))

## 2. Parse fields out of the prompt

The ads look structured, but they are still strings. We regex out brand, model, year, mileage, fuel, transmission, then treat `completion` as the integer EGP price.

Rows that fail to parse are dropped (logged as `parse_failed`).

In [ ]:
parsed, n_fail = parse_frame(raw)
print(f"Parsed {len(parsed):,}   failed {n_fail:,}")
display(parsed.head(3))
print("\nBrands:", parsed["brand"].nunique(), "  Models:", parsed["model"].nunique())
print("Year range:", parsed["year"].min(), "–", parsed["year"].max())
print("Price range: EGP", f"{parsed['price'].min():,.0f}", "–", f"{parsed['price'].max():,.0f}")
print("Fuel:", parsed["fuel"].value_counts().to_dict())
print("Transmission:", parsed["transmission"].value_counts().to_dict())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(parsed["price"] / 1e6, bins=40, color="steelblue")
axes[0].set_title("Price (million EGP)")
axes[1].hist(parsed["year"], bins=range(1990, 2028), color="darkseagreen")
axes[1].set_title("Year")
axes[2].hist(parsed["mileage"] / 1000, bins=40, color="goldenrod")
axes[2].set_title("Mileage (thousand km)")
plt.tight_layout()
plt.show()

print("Top brands:\n", parsed["brand"].value_counts().head(15).to_string())
print("\nBrand == Other:", (parsed["brand"].str.casefold() == "other").sum())
print("Model == Other:", (parsed["model"].str.casefold() == "other").sum())
print("Mileage == 0:", (parsed["mileage"] == 0).sum())
print("Mileage == 0 and year <= 2023:", ((parsed["mileage"] == 0) & (parsed["year"] <= 2023)).sum())
print("gen_suspect flags (kept for now):", int(parsed["gen_suspect"].sum()))

## 3. Hard filters (applied one rule at a time in `apply_hard_filters`)

| Rule | Drop when |
|---|---|
| `other_brand_or_model` | Brand or Model is `Other` |
| `year_out_of_range` | Year not in 1990–2027 |
| `mileage_out_of_range` | Mileage &lt; 0 or &gt; 800,000 km |
| `price_global_clip` | Price &lt; 50,000 or &gt; 20,000,000 EGP |
| `zero_km_old_car` | Mileage is 0 **and** year ≤ 2023 |

0 km on a 2025/2026 car can be a new listing. 0 km on a 1998 Fiat is missing data.

Generation mismatches (Wrangler 4xe in 2017, Octavia A8 in 2015) are **flagged** as `gen_suspect`, not dropped in v1.

In [ ]:
kept, dropped_hard = apply_hard_filters(parsed)
print("Hard-filter drop counts:")
print(dropped_hard["drop_reason"].value_counts().to_string() if len(dropped_hard) else "(none)")
print(f"\nStill kept: {len(kept):,} / {len(parsed):,}")

kept, n_dup = drop_exact_duplicates(kept)
print(f"Exact prompt+completion duplicates dropped: {n_dup:,}")
print(f"After dedup: {len(kept):,}")

## 4. Within-group price outliers

A 2023 Mercedes C 180 at 300,000 EGP is not a global clip problem — it is absurd **relative to other Mercedes C-class cars**.

- If `(brand, model, year)` has ≥ 8 rows: drop Tukey IQR outliers (`Q1 − 1.5 IQR`, `Q3 + 1.5 IQR`)
- Else if `(brand, model)` has ≥ 12 rows: drop prices outside `0.25×–4×` the model median

In [ ]:
kept, dropped_out = apply_price_outliers(kept)
print(f"Price outliers dropped: {len(dropped_out):,}")
if len(dropped_out):
    show = dropped_out.sort_values("price").head(8)[
        ["brand", "model", "year", "mileage", "price"]
    ]
    display(show)
print(f"\nClean rows: {len(kept):,}")
print("gen_suspect still in the clean set:", int(kept["gen_suspect"].sum()))

## 5. Train / val / test split

- **test = 200** (frozen for the Week 7-style Tester)
- **val = 500**
- **train = the rest**
- Stratify by brand (rare brands with &lt; 5 rows are grouped)
- Seed `42`

`LITE_MODE` later uses 8,000 of these train rows, but **the same 200 test cars**.

In [ ]:
train, val, test = split_frame(kept)
train_lite = lite_train(train, n=LITE_TRAIN_SIZE)

print(f"train      {len(train):,}")
print(f"train_lite {len(train_lite):,}   (LITE_MODE)")
print(f"val        {len(val):,}     (target {VAL_SIZE})")
print(f"test       {len(test):,}     (target {TEST_SIZE})")
print("\nTest brands (top):")
print(test["brand"].value_counts().head(10).to_string())
print("\nExample test prompt:\n")
print(test.iloc[0]["prompt"])
print("completion:", test.iloc[0]["completion"])

## 6. Save locally (no Hub push)

Writes parquet + a Hugging Face `DatasetDict` under `data/`. Those files are gitignored.

The next cell is **commented on purpose**. Uncomment `CarItem.push_prompts_to_hub(...)` only after you explicitly ask to publish.

In [ ]:
report = {
    "source_dataset": SOURCE_DATASET,
    "source_rows": int(len(raw)),
    "parse_failed": int(n_fail),
    "hard_drop": dropped_hard["drop_reason"].value_counts().to_dict() if len(dropped_hard) else {},
    "exact_duplicates": int(n_dup),
    "price_outliers": int(len(dropped_out)),
    "kept": int(len(kept)),
    "gen_suspect_kept": int(kept["gen_suspect"].sum()),
    "splits": {
        "train": int(len(train)),
        "train_lite": int(len(train_lite)),
        "val": int(len(val)),
        "test": int(len(test)),
    },
}
out = save_splits(train, val, test, report, lite=train_lite, data_dir=ROOT / "data")
print("Wrote:", out)
print(pd.Series(report).to_string())

# Do not push unless asked:
# from pricer.items import CarItem
# from pricer.prep import to_items
# CarItem.push_prompts_to_hub("YOUR_USER/egyptian-cars-prompts", to_items(train), to_items(val), to_items(test))